# The Story That Was Passed On: Wan 2.2 TI2V-5B Video Synthesis
**Production Pipeline: Milestone 5 (Kaggle Environment)**

This notebook executes the video generation stage of the pipeline using `Wan-AI/Wan2.2-TI2V-5B-Diffusers` (`WanImageToVideoPipeline`).
- **Target Shots**: Filtered strictly to shots where `use_video == true` (`shot_02, 03, 04, 06, 07, 09`).
- **Resolution**: Draft at `832x480` (121 frames @ 24 fps = ~5.04s) for fast iteration, with an optional `1280x704` high-res cell for key close-ups.
- **Memory Optimization**: Utilizes `torch.float16` and `enable_model_cpu_offload()` to run stably on Kaggle GPUs (T4 x2 or P100 / L4).
- **Telemetry**: Appends exact generation telemetry and GPU minutes to `kaggle_runs.jsonl`.

In [ ]:
# Step 1: Install dependencies
!pip install -q --upgrade pip
!pip install -q diffusers transformers accelerate torch torchvision imageio[ffmpeg] sentencepiece protobuf

In [ ]:
# Step 2: Locate Storyboard and Keyframe Inputs
import json
import os
from pathlib import Path

# Support both Kaggle dataset paths and local workspace checkouts
def find_input_paths():
    candidates = [
        Path('/kaggle/input/the-story-that-was-passed-on'),
        Path('/kaggle/input/storyboard-keyframes'),
        Path('.'),
        Path('..'),
    ]
    for p in candidates:
        sb = p / 'storyboard.json'
        if sb.is_file():
            kf = p / 'outputs' / 'keyframes'
            if not kf.is_dir():
                kf = p / 'keyframes'
            return sb, kf
    raise FileNotFoundError('Could not locate storyboard.json in standard Kaggle input directories.')

storyboard_path, keyframes_dir = find_input_paths()
print(f'Storyboard located: {storyboard_path}')
print(f'Keyframes directory: {keyframes_dir}')

with open(storyboard_path, 'r', encoding='utf-8') as f:
    storyboard = json.load(f)

video_shots = [s for s in storyboard['shots'] if s.get('use_video', False)]
print(f'Total video shots to synthesize: {len(video_shots)} -> {[s["id"] for s in video_shots]}')

output_shots_dir = Path('/kaggle/working/outputs/shots')
output_shots_dir.mkdir(parents=True, exist_ok=True)
runs_log_path = Path('/kaggle/working/outputs/kaggle_runs.jsonl')

In [ ]:
# Step 3: Initialize Wan 2.2 TI2V-5B Pipeline
import torch
from diffusers import WanImageToVideoPipeline
from diffusers.utils import export_to_video

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

print('Loading Wan2.2-TI2V-5B model in float16 with CPU offload...')
pipe = WanImageToVideoPipeline.from_pretrained(
    'Wan-AI/Wan2.2-TI2V-5B-Diffusers',
    torch_dtype=torch.float16,
)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()  # Prevents T4 16GB VRAM OOM during VAE decode
print('Wan 2.2 TI2V-5B loaded with VAE tiling enabled!')

In [ ]:
# Step 3B: Quick 1-Shot Smoke Test (Validates VRAM & pipeline before running full 6 shots)
from PIL import Image
test_shot = video_shots[0]
test_kf = keyframes_dir / f"{test_shot['id']}.png"
print(f"Running 1-shot smoke test on {test_shot['id']} at 832x480 (17 frames draft)...\n")
test_init = Image.open(test_kf).convert('RGB').resize((832, 480)) if test_kf.is_file() else Image.new('RGB', (832, 480))
test_out = pipe(
    image=test_init,
    prompt=test_shot.get('motion_prompt', ''),
    num_frames=17,
    height=480,
    width=832,
    num_inference_steps=10,
)
print('Smoke test succeeded! Ready for full 6-shot batch generation.')

In [ ]:
# Step 4: Batch Video Generation - Draft Resolution (832x480, 121 frames @ 24 fps)
import time
import uuid
from datetime import datetime, timezone
from PIL import Image

WIDTH = 832
HEIGHT = 480
NUM_FRAMES = 121  # ~5.04 seconds at 24 fps
FPS = 24
STEPS = 30

print(f'Starting generation for {len(video_shots)} shots at {WIDTH}x{HEIGHT} (121 frames, {STEPS} steps)...')

for shot in video_shots:
    shot_id = shot['id']
    out_mp4 = output_shots_dir / f'{shot_id}.mp4'
    
    # Locate source keyframe
    kf_file = keyframes_dir / f'{shot_id}.png'
    if not kf_file.is_file():
        print(f'WARNING: Keyframe {kf_file} not found; skipping {shot_id}')
        continue
    
    init_image = Image.open(kf_file).convert('RGB').resize((WIDTH, HEIGHT))
    prompt = f"{shot.get('motion_prompt', '')} {shot.get('image_prompt', '')}"
    seed = shot.get('seed', 1001)
    generator = torch.Generator(device='cuda').manual_seed(seed)
    
    print(f'\n[Generating {shot_id}] Prompt: {prompt[:80]}... | Seed: {seed}')
    start_time = time.perf_counter()
    
    output = pipe(
        image=init_image,
        prompt=prompt,
        negative_prompt='distortion, blur, low quality, jerky camera, artifacts, watermarks, subtitles',
        num_frames=NUM_FRAMES,
        height=HEIGHT,
        width=WIDTH,
        num_inference_steps=STEPS,
        generator=generator,
    )
    
    latency = round(time.perf_counter() - start_time, 2)
    gpu_minutes = round(latency / 60.0, 3)
    
    export_to_video(output.frames[0], str(out_mp4), fps=FPS)
    print(f'[{shot_id}] Saved -> {out_mp4.name} ({out_mp4.stat().st_size / 1e6:.2f} MB in {latency}s / {gpu_minutes} GPU min)')
    
    # Telemetry logging to kaggle_runs.jsonl
    log_entry = {
        'run_id': f'wan22_{shot_id}_{uuid.uuid4().hex[:8]}',
        'stage': 'video',
        'shot_id': shot_id,
        'provider': 'kaggle_wan22',
        'model': 'Wan-AI/Wan2.2-TI2V-5B-Diffusers',
        'seed': seed,
        'parameters': {
            'width': WIDTH,
            'height': HEIGHT,
            'num_frames': NUM_FRAMES,
            'fps': FPS,
            'steps': STEPS,
        },
        'output': str(out_mp4),
        'latency_seconds': latency,
        'gpu_minutes': gpu_minutes,
        'status': 'success',
        'timestamp': datetime.now(timezone.utc).isoformat(),
    }
    with open(runs_log_path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(log_entry) + '\n')

### Optional Step 4B: High-Resolution Close-Up Render (1280x704)
Run this cell for high-impact close-up shots (`shot_03`, `shot_04`, `shot_07`) where facial expressions and subtle motions require full native 720p fidelity.

In [ ]:
# Optional High-Res close-up cell (1280x704)
HIRES_SHOTS = ['shot_03', 'shot_04', 'shot_07']
HIRES_W = 1280
HIRES_H = 704

for shot in [s for s in video_shots if s['id'] in HIRES_SHOTS]:
    shot_id = shot['id']
    out_mp4 = output_shots_dir / f'{shot_id}_hires.mp4'
    kf_file = keyframes_dir / f'{shot_id}.png'
    
    if not kf_file.is_file():
        continue
    
    init_image = Image.open(kf_file).convert('RGB').resize((HIRES_W, HIRES_H))
    prompt = f"{shot.get('motion_prompt', '')} {shot.get('image_prompt', '')}"
    seed = shot.get('seed', 1001)
    generator = torch.Generator(device='cuda').manual_seed(seed)
    
    print(f'\n[High-Res 1280x704: {shot_id}] Seed: {seed}...')
    start_time = time.perf_counter()
    
    output = pipe(
        image=init_image,
        prompt=prompt,
        num_frames=NUM_FRAMES,
        height=HIRES_H,
        width=HIRES_W,
        num_inference_steps=35,
        generator=generator,
    )
    latency = round(time.perf_counter() - start_time, 2)
    export_to_video(output.frames[0], str(out_mp4), fps=FPS)
    print(f'[{shot_id}_hires] Saved -> {out_mp4.name} in {latency}s')

In [ ]:
# Step 5: Archive Outputs for Pipeline Ingestion
import shutil

archive_path = '/kaggle/working/wan22_video_outputs.zip'
shutil.make_archive('/kaggle/working/wan22_video_outputs', 'zip', '/kaggle/working/outputs')
print(f'Outputs packaged to: {archive_path} ({os.path.getsize(archive_path) / 1e6:.2f} MB)')
print('Download this zip and extract into the local repository under outputs/shots/!')